In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import joblib

# =========================
# 0) 全局：尽量屏蔽警告
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的特征文件
FEATURE_FILE_GLOB = "../Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

# Optuna 搜索输出的结果文件
OPTUNA_RESULTS_CSV = "../optuna_singlemodel_XGB_30trials_5fold_results.csv"

# AD 搜索前需要保存的五折运行数据
SAVE_DIR = "./best_xgb_5fold_saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# 是否清理旧的 fold_*.npz，建议 True，避免新旧折数据混用
CLEAN_OLD_FOLD_FILES = True

OUT_FOLDS_CSV = os.path.join(SAVE_DIR, "xgb_bestparams_5fold_metrics.csv")
OUT_MEANCI_CSV = os.path.join(SAVE_DIR, "xgb_bestparams_5fold_mean_ci95.csv")
OUT_MODEL_FILE = os.path.join(SAVE_DIR, "xgb_bestparams_fullfit.joblib")
OUT_OOF_FILE = os.path.join(SAVE_DIR, "oof_predictions.npz")
OUT_PARAM_JSON = os.path.join(SAVE_DIR, "best_params.json")

BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",  # 需要 xgboost >= 2.0
)

# =========================
# 2) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")

    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree，需要 xgboost >= 2.0。"
        )

# =========================
# 3) 读取特征文件
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def load_Xy_from_transformed_morgan(xlsx_path: str):
    """
    固定列分配：
    第 1 列：SMILES / StdSMILES
    第 2–25 列：24 个气味描述符，作为 y
    第 26 列到最后：全部分子特征，作为 X
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件只有 {df.shape[1]} 列，但至少需要 26 列。"
        )

    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别到 {len(label_cols)} 个：{label_cols}"
        )

    if len(feature_cols) == 0:
        raise ValueError("未识别到特征列。请确认第 26 列之后为分子特征。")

    smiles = df[smiles_col].astype(str).values
    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(np.int8).values

    print("[INFO] SMILES column:", smiles_col)
    print("[INFO] Label columns:", label_cols)
    print("[INFO] First 5 feature columns:", feature_cols[:5])

    return X, y, smiles, feature_cols, label_cols, df

# =========================
# 4) 指标
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    """
    兼容不同版本 XGBClassifier 的 predict_proba 输出。
    """
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)

    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)

    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)

    raise ValueError(f"无法解析 predict_proba 输出形状：{p.shape}")

# =========================
# 5) 近似分层 5 折
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)

    if len(uniq) <= 15:
        return card

    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(
        r,
        q=min(n_bins, len(np.unique(r))),
        labels=False,
        duplicates="drop"
    )
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )
    return list(skf.split(X, strat_y))

# =========================
# 6) 读取 Optuna 最优超参
# =========================
def sanitize_xgb_params(params: dict) -> dict:
    p = dict(params)

    int_keys = ["n_estimators", "max_depth", "n_jobs", "random_state"]
    for k in int_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = int(float(p[k]))
            except Exception:
                pass

    float_keys = [
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "min_child_weight",
        "reg_lambda",
        "reg_alpha",
        "gamma",
    ]
    for k in float_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = float(p[k])
            except Exception:
                pass

    return p

def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)

    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = {
        "trial",
        "AUPRC_macro",
        "AUROC_macro",
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
    }

    params = {}
    for k, v in best_row.items():
        if k not in drop_cols:
            params[k] = v

    final_params = dict(BASE_XGB_PARAMS)
    final_params.update(params)
    final_params = sanitize_xgb_params(final_params)

    return final_params

# =========================
# 7) mean ± CI95
# =========================
def mean_ci95(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    n = len(arr)

    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=1)) if n >= 2 else 0.0
    se = std / np.sqrt(n) if n > 0 else np.nan

    try:
        import scipy.stats as st
        tcrit = float(st.t.ppf(0.975, df=n - 1)) if n >= 2 else 1.96
    except Exception:
        tcrit = 1.96

    half = tcrit * se if n >= 2 else 0.0
    return mean, mean - half, mean + half, n

def format_mean_pm_ci95(mean, lo, hi):
    if np.isnan(mean):
        return "nan"
    half = (hi - lo) / 2.0
    return f"{mean:.3f} ± {half:.3f}"

def summarize_mean_ci95(folds_df: pd.DataFrame):
    metric_cols = [
        "Accuracy_macro",
        "Precision_macro",
        "Recall_macro",
        "Specificity_macro",
        "AUROC_macro",
        "AUPRC_macro",
    ]

    rows = []
    for col in metric_cols:
        mean, lo, hi, n = mean_ci95(folds_df[col].values)
        rows.append({
            "metric": col,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_folds": n,
            "mean ± CI95": format_mean_pm_ci95(mean, lo, hi),
        })

    return pd.DataFrame(rows)

# =========================
# 8) 清理旧保存文件
# =========================
def clean_old_fold_files(save_dir):
    old_npz = glob.glob(os.path.join(save_dir, "fold_*.npz"))
    for p in old_npz:
        os.remove(p)

    old_files = [
        OUT_OOF_FILE,
        OUT_PARAM_JSON,
        OUT_FOLDS_CSV,
        OUT_MEANCI_CSV,
    ]

    for p in old_files:
        if os.path.exists(p):
            os.remove(p)

# =========================
# 9) 五折训练、评估并保存 AD 所需数据
# =========================
def cv_eval_and_save_bestparams(
    X,
    y,
    smiles,
    feature_cols,
    label_cols,
    folds,
    params,
    save_dir,
    thresh=0.5,
):
    n_samples = X.shape[0]
    n_labels = y.shape[1]

    fold_rows = []

    oof_prob = np.zeros((n_samples, n_labels), dtype=np.float32)
    oof_fold = np.zeros(n_samples, dtype=np.int32)

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        print(f"\n========== FOLD {fold_id} / {len(folds)} ==========")

        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob_va = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob_va, thresh=thresh)
        m["fold"] = fold_id
        m["n_train"] = int(len(tr_idx))
        m["n_valid"] = int(len(va_idx))
        fold_rows.append(m)

        oof_prob[va_idx] = y_prob_va
        oof_fold[va_idx] = fold_id

        save_path = os.path.join(save_dir, f"fold_{fold_id:02d}.npz")

        np.savez_compressed(
            save_path,

            # AD 搜索脚本必须读取的 5 个字段
            X_tr=X_tr.astype(np.float32),
            y_tr=y_tr.astype(np.int8),
            X_va=X_va.astype(np.float32),
            y_va=y_va.astype(np.int8),
            y_prob_va=y_prob_va.astype(np.float32),

            # 额外保存，便于后续核查与复现实验
            tr_idx=tr_idx.astype(np.int64),
            va_idx=va_idx.astype(np.int64),
            smiles_tr=smiles[tr_idx].astype(object),
            smiles_va=smiles[va_idx].astype(object),
            feature_cols=np.array(feature_cols, dtype=object),
            label_cols=np.array(label_cols, dtype=object),
            threshold=np.array([thresh], dtype=np.float32),
            random_seed=np.array([RANDOM_SEED], dtype=np.int32),
        )

        print(
            f"[FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | "
            f"AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | "
            f"P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | "
            f"Spec={m['Specificity_macro']:.6f}"
        )

        print(f"[SAVED] {save_path}")

    folds_df = pd.DataFrame(fold_rows).sort_values("fold")

    np.savez_compressed(
        OUT_OOF_FILE,
        y_true=y.astype(np.int8),
        y_prob=oof_prob.astype(np.float32),
        fold=oof_fold.astype(np.int32),
        smiles=smiles.astype(object),
        feature_cols=np.array(feature_cols, dtype=object),
        label_cols=np.array(label_cols, dtype=object),
        threshold=np.array([thresh], dtype=np.float32),
        random_seed=np.array([RANDOM_SEED], dtype=np.int32),
    )

    print(f"\n[SAVED] OOF predictions -> {OUT_OOF_FILE}")

    return folds_df

# =========================
# 10) 主流程
# =========================
def main():
    check_xgb_version()

    if CLEAN_OLD_FOLD_FILES:
        clean_old_fold_files(SAVE_DIR)
        print(f"[INFO] Old fold files cleaned in: {SAVE_DIR}")

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, smiles, feature_cols, label_cols, _df_all = load_Xy_from_transformed_morgan(feature_file)

    print(
        f"[INFO] X shape={X.shape} | "
        f"y shape={y.shape} | "
        f"features={len(feature_cols)} | "
        f"labels={len(label_cols)}"
    )

    best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)

    print("\n[INFO] Loaded best params from Optuna CSV:")
    print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

    with open(OUT_PARAM_JSON, "w", encoding="utf-8") as f:
        json.dump(
            {
                "best_params": best_params,
                "random_seed": RANDOM_SEED,
                "n_splits": N_SPLITS,
                "threshold": THRESH,
                "feature_file": feature_file,
                "optuna_results_csv": OPTUNA_RESULTS_CSV,
                "n_samples": int(X.shape[0]),
                "n_features": int(X.shape[1]),
                "n_labels": int(y.shape[1]),
                "label_cols": list(label_cols),
                "feature_cols": list(feature_cols),
            },
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    print(f"[SAVED] params metadata -> {OUT_PARAM_JSON}")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"\n[INFO] Prepared fixed {N_SPLITS}-fold splits.")

    # 五折训练、评估、保存每折 AD 所需数据
    folds_df = cv_eval_and_save_bestparams(
        X=X,
        y=y,
        smiles=smiles,
        feature_cols=feature_cols,
        label_cols=label_cols,
        folds=folds,
        params=best_params,
        save_dir=SAVE_DIR,
        thresh=THRESH,
    )

    folds_df.to_csv(OUT_FOLDS_CSV, index=False, encoding="utf-8-sig")
    print(f"\n[SAVED] fold metrics -> {OUT_FOLDS_CSV}")

    sum_df = summarize_mean_ci95(folds_df)
    sum_df.to_csv(OUT_MEANCI_CSV, index=False, encoding="utf-8-sig")
    print(f"[SAVED] mean CI95 -> {OUT_MEANCI_CSV}")

    print("\n========== 5-FOLD MEAN ± CI95 ==========")
    for _, r in sum_df.iterrows():
        print(f"{r['metric']}: {r['mean ± CI95']}")

    # 全量训练并保存最终模型
    print("\n[INFO] Fitting full model on all data...")
    clf_full = xgb.XGBClassifier(**best_params)
    clf_full.fit(X, y)

    joblib.dump(
        {
            "model": clf_full,
            "params": best_params,
            "feature_cols": feature_cols,
            "label_cols": label_cols,
            "threshold": THRESH,
            "random_seed": RANDOM_SEED,
            "feature_file": feature_file,
        },
        OUT_MODEL_FILE
    )

    print(f"[SAVED] full-fit model -> {OUT_MODEL_FILE}")

    print("\n[DONE] AD 搜索所需五折数据已保存到：")
    print(SAVE_DIR)
    print("\n后续 AD 搜索脚本中的 fold_dir 保持为：")
    print('fold_dir = "./best_xgb_5fold_saved"')

if __name__ == "__main__":
    main()

[INFO] Old fold files cleaned in: ./best_xgb_5fold_saved
[INFO] Using feature file: ../Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: ['alcoholic', 'aldehydic', 'almond', 'aromatic', 'burnt', 'cabbage', 'cheesy', 'cherry', 'chocolate', 'ethereal', 'fishy', 'fruity', 'garlic', 'grassy', 'green', 'ketonic', 'musty', 'pungent', 'sharp', 'solvent', 'sour', 'sulfurous', 'sweaty', 'sweet']
[INFO] First 5 feature columns: ['FG: Sulfone(–SO2–)', 'FG: Disulfide(S–S)', 'FG: [CX3](=O)[#6][#6]', 'FG: NX3-H2-H1-NC-O', 'FG: Thiol(–SH)']
[INFO] X shape=(3756, 2595) | y shape=(3756, 24) | features=2595 | labels=24

[INFO] Loaded best params from Optuna CSV:
{
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "tree_method": "hist",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": 0,
  "multi_strategy": "multi_output_tree",
  "n_estimators": 433,
  "max_depth": 7,
  "learning_rate": 0.0350057872293877,
  "subsample": 0.994715

In [2]:
# -*- coding: utf-8 -*-
import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# =========================
# 配置
# =========================
fold_dir   = "./best_xgb_5fold_saved"
n_folds    = 5

GRID_SIZE  = 20
MIN_IN_SAMPLES = 30
Q_LOW, Q_HIGH = 0.05, 0.95

k_list       = np.arange(0.05, 0.15, 0.05)
a_list       = [10, 100, 500, 1000, 10000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

LABEL_DIFF_MODE = "jaccard"
K_SWD_CAP = 300

OUT_CSV = os.path.join(fold_dir, "AD_SAL_multilabel_search_VERBOSE_live.csv")

# =========================
# 工具
# =========================
def l2_normalize(X, eps=1e-12):
    X = X.astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def cosine_sim_query(Xq_n, Xtr_n):
    return np.clip(Xq_n @ Xtr_n.T, 0.0, 1.0).astype(np.float32)

def compute_weights(sim_vals, a, epsilon):
    s = np.clip(sim_vals, epsilon, 1.0)
    return np.exp(-a * (1.0 - s) / s).astype(np.float32)

def pairwise_label_diff_scalar(y_t, y_nei, mode="jaccard", eps=1e-12):
    y_t = y_t.astype(np.int8, copy=False)
    y_nei = y_nei.astype(np.int8, copy=False)
    if mode == "hamming":
        return np.mean(np.abs(y_nei - y_t[None, :]), axis=1).astype(np.float32)
    if mode == "jaccard":
        inter = np.sum((y_nei == 1) & (y_t[None, :] == 1), axis=1).astype(np.float32)
        union = np.sum((y_nei == 1) | (y_t[None, :] == 1), axis=1).astype(np.float32)
        j = inter / (union + eps)
        return (1.0 - j).astype(np.float32)
    raise ValueError

def per_label_ap(y_true, y_prob):
    y_true = np.asarray(y_true).astype(np.int8)
    y_prob = np.asarray(y_prob).astype(np.float32)
    L = y_true.shape[1]
    ap = np.full(L, np.nan, dtype=float)
    for j in range(L):
        yt = y_true[:, j]
        if len(np.unique(yt)) < 2:
            continue
        ap[j] = average_precision_score(yt, y_prob[:, j])
    return ap

def macro_auprc_with_baseline_fill(y_true_in, y_prob_in, ap_baseline, label_idx):
    ap_in = per_label_ap(y_true_in, y_prob_in)
    ap_filled = ap_in.copy()
    nan_mask = np.isnan(ap_filled)
    ap_filled[nan_mask] = ap_baseline[nan_mask]
    return float(np.mean(ap_filled[label_idx]))

def load_fold_npz(fold_id: int):
    path = os.path.join(fold_dir, f"fold_{fold_id:02d}.npz")
    z = np.load(path, allow_pickle=True)
    return z["X_tr"], z["y_tr"], z["X_va"], z["y_va"], z["y_prob_va"]

def append_row_to_csv(row: dict, csv_path: str):
    df = pd.DataFrame([row])
    header = not os.path.exists(csv_path)
    df.to_csv(csv_path, mode="a", index=False, encoding="utf-8-sig", header=header)

def compute_SWD_topk_with_progress(Xtr, ytr, k_swd_eff, a, eps, fold_id, print_every=500):
    """
    这里会构造 (N,N) 相似度矩阵 -> 最耗时
    加了进度打印，保证你知道在跑
    """
    Xn = l2_normalize(Xtr)
    sim = cosine_sim_query(Xn, Xn)  # (N,N)

    N = ytr.shape[0]
    SWD = np.zeros(N, dtype=np.float32)
    k_eff = max(1, min(k_swd_eff, N-1))

    t0 = time.time()
    for t in range(N):
        sims_t = sim[t]
        idxs = np.argpartition(sims_t, -(k_eff + 1))[-(k_eff + 1):]
        idxs = idxs[idxs != t]
        if idxs.size > 0:
            sv = sims_t[idxs]
            w  = compute_weights(sv, a, eps)
            diffs = pairwise_label_diff_scalar(ytr[t], ytr[idxs], mode=LABEL_DIFF_MODE)
            SWD[t] = float((w * sv * diffs).sum() / (w.sum() + eps))

        if (t + 1) % print_every == 0 or (t + 1) == N:
            dt = time.time() - t0
            print(f"    [SWD fold {fold_id}] {t+1}/{N} done ({dt:.1f}s)", flush=True)

    return SWD, Xn

def compute_rho_IA_given_SWD(Xtr_n, SWD, Xva, k_eff, a, eps):
    Xva_n = l2_normalize(Xva)
    sim_qt = cosine_sim_query(Xva_n, Xtr_n)  # (M,N)
    M = Xva.shape[0]
    rho = np.zeros(M, dtype=np.float32)
    IA  = np.zeros(M, dtype=np.float32)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k_eff)[-k_eff:]
        sk = sims[idxk]
        w  = compute_weights(sk, a, eps)
        rho[i] = float(w.mean())
        IA[i]  = float((w * SWD[idxk]).sum() / (w.sum() + eps))
    return rho, IA

# =========================
# 主流程
# =========================
def main():
    # 清理旧输出（如果你想续跑，注释掉）
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    # 预读 folds
    folds = []
    for f in range(1, n_folds+1):
        X_tr, y_tr, X_va, y_va, y_prob_va = load_fold_npz(f)
        folds.append((X_tr, y_tr, X_va, y_va, y_prob_va))

    total_jobs = len(a_list) * len(epsilon_list) * len(k_list)
    job_id = 0
    t_global = time.time()

    for a_weight in a_list:
        for eps in epsilon_list:
            print(f"\n=== START (a={a_weight}, eps={eps:.0e}) ===", flush=True)

            # 先算 SWD（并打印进度）
            swd_cache = []
            for fold_id, (X_tr, y_tr, X_va, y_va, y_prob_va) in enumerate(folds, start=1):
                k_swd_eff = min(max(int(0.1 * X_tr.shape[0]), 1), K_SWD_CAP)
                print(f"  -> computing SWD for fold {fold_id}, k_swd_eff={k_swd_eff}", flush=True)
                SWD, Xtr_n = compute_SWD_topk_with_progress(
                    X_tr, y_tr, k_swd_eff, a_weight, eps, fold_id=fold_id, print_every=500
                )
                swd_cache.append((SWD, Xtr_n, k_swd_eff))

            # 再对每个 k_ratio 输出一行结果
            for k_ratio in k_list:
                job_id += 1
                t_job = time.time()

                fold_cache = []
                all_rho, all_IA = [], []

                for idx, (X_tr, y_tr, X_va, y_va, y_prob_va) in enumerate(folds):
                    SWD, Xtr_n, k_swd_eff = swd_cache[idx]
                    k_eff = max(int(float(k_ratio) * X_tr.shape[0]), 1)
                    k_eff = min(k_eff, X_tr.shape[0])

                    rho_s, IA = compute_rho_IA_given_SWD(Xtr_n, SWD, X_va, k_eff, a_weight, eps)

                    fold_cache.append((rho_s, IA, y_va, y_prob_va))
                    all_rho.append(rho_s); all_IA.append(IA)

                all_rho = np.concatenate(all_rho)
                all_IA  = np.concatenate(all_IA)

                # baseline & 固定标签集合
                Y_all = np.concatenate([fc[2] for fc in fold_cache], axis=0)
                P_all = np.concatenate([fc[3] for fc in fold_cache], axis=0)
                ap_baseline = per_label_ap(Y_all, P_all)
                label_idx = np.array([j for j in range(Y_all.shape[1]) if not np.isnan(ap_baseline[j])], dtype=int)

                if label_idx.size == 0:
                    best_ap = np.nan
                    best_cov = np.nan
                    best_rho_th = np.nan
                    best_ia_th = np.nan
                else:
                    rho_cuts = np.quantile(all_rho, np.linspace(Q_LOW, Q_HIGH, GRID_SIZE))
                    IA_cuts  = np.quantile(all_IA,  np.linspace(Q_LOW, Q_HIGH, GRID_SIZE))

                    best_ap = -np.inf
                    best_cov = 0.0
                    best_rho_th = None
                    best_ia_th = None

                    for ia_th in IA_cuts:
                        for rho_th in rho_cuts:
                            s, cnt = 0.0, 0
                            cov_sum = 0.0
                            for (rho_s, IA, y_va, y_prob_va) in fold_cache:
                                in_m = (rho_s >= rho_th) & (IA <= ia_th)
                                n_in = int(in_m.sum())
                                if n_in < MIN_IN_SAMPLES:
                                    continue
                                cov_sum += n_in / len(rho_s)
                                ap_in = macro_auprc_with_baseline_fill(
                                    y_va[in_m], y_prob_va[in_m],
                                    ap_baseline, label_idx
                                )
                                s += ap_in
                                cnt += 1
                            if cnt > 0:
                                mean_ap = s / cnt
                                mean_cov = cov_sum / cnt
                                if mean_ap > best_ap:
                                    best_ap = mean_ap
                                    best_cov = mean_cov
                                    best_rho_th = float(rho_th)
                                    best_ia_th  = float(ia_th)

                    if not np.isfinite(best_ap):
                        best_ap = np.nan
                        best_cov = np.nan
                        best_rho_th = np.nan
                        best_ia_th = np.nan

                row = {
                    "job_id": job_id,
                    "job_total": total_jobs,
                    "a": int(a_weight),
                    "eps": float(eps),
                    "k_ratio": float(k_ratio),
                    "GRID_SIZE": int(GRID_SIZE),
                    "MIN_IN_SAMPLES": int(MIN_IN_SAMPLES),
                    "K_SWD_CAP": int(K_SWD_CAP),
                    "best_mean_AUPRC_in": float(best_ap) if np.isfinite(best_ap) else np.nan,
                    "best_mean_coverage_in": float(best_cov) if np.isfinite(best_cov) else np.nan,
                    "best_rho_th": best_rho_th,
                    "best_IA_th": best_ia_th,
                    "label_diff_mode": LABEL_DIFF_MODE,
                    "elapsed_sec_this_job": round(time.time() - t_job, 3),
                    "elapsed_min_total": round((time.time() - t_global) / 60.0, 2),
                }

                append_row_to_csv(row, OUT_CSV)

                print(
                    f"[{job_id}/{total_jobs}] a={a_weight:<5} eps={eps:.0e} k={k_ratio:>4.2f} | "
                    f"AUPRC_in={row['best_mean_AUPRC_in']:.4f} cov={row['best_mean_coverage_in']:.3f} | "
                    f"(rho_th={row['best_rho_th']:.4g}, IA_th={row['best_IA_th']:.4g}) | "
                    f"{row['elapsed_sec_this_job']}s",
                    flush=True
                )

    print("\n[SAVED]", OUT_CSV, flush=True)

if __name__ == "__main__":
    main()


=== START (a=10, eps=1e-03) ===
  -> computing SWD for fold 1, k_swd_eff=300
    [SWD fold 1] 500/3004 done (0.1s)
    [SWD fold 1] 1000/3004 done (0.1s)
    [SWD fold 1] 1500/3004 done (0.2s)
    [SWD fold 1] 2000/3004 done (0.2s)
    [SWD fold 1] 2500/3004 done (0.3s)
    [SWD fold 1] 3000/3004 done (0.3s)
    [SWD fold 1] 3004/3004 done (0.3s)
  -> computing SWD for fold 2, k_swd_eff=300
    [SWD fold 2] 500/3005 done (0.1s)
    [SWD fold 2] 1000/3005 done (0.1s)
    [SWD fold 2] 1500/3005 done (0.2s)
    [SWD fold 2] 2000/3005 done (0.2s)
    [SWD fold 2] 2500/3005 done (0.3s)
    [SWD fold 2] 3000/3005 done (0.3s)
    [SWD fold 2] 3005/3005 done (0.4s)
  -> computing SWD for fold 3, k_swd_eff=300
    [SWD fold 3] 500/3005 done (0.1s)
    [SWD fold 3] 1000/3005 done (0.1s)
    [SWD fold 3] 1500/3005 done (0.2s)
    [SWD fold 3] 2000/3005 done (0.2s)
    [SWD fold 3] 2500/3005 done (0.3s)
    [SWD fold 3] 3000/3005 done (0.4s)
    [SWD fold 3] 3005/3005 done (0.4s)
  -> computing S

In [ ]:
 a=100   eps=1e-06 k=0.05 | AUPRC_in=0.5453 cov=0.252 | (rho_th=8.445e-08, IA_th=0.4438) 